In [1]:
# %%
# =============================================================================
# NOTEBOOK: 04_results_viz_D_TDC.ipynb
# Paper D (TDC) — headline figures: "simplicity wins"
#   (A) strategy comparison: AUC (CIs overlap) vs cost (elaborate = costly,
#       not better)
#   (B) model strength drives performance; best text (gpt-5.4) beats numeric
#
# No API calls; reuses D_full_strategy_comparison.csv and D_baselines.csv.
# Data: text-based bankruptcy dataset, Mendeley DOI 10.17632/stf3kg7fw3
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
TAB = ROOT / "artifacts" / "tables"
FIG = ROOT / "artifacts" / "figures"
FIG.mkdir(parents=True, exist_ok=True)


def rel(p):
    try: return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError: return Path(p).name


SRC = "Data: text-based bankruptcy dataset, Mendeley DOI 10.17632/stf3kg7fw3."
NUMERIC_BASELINE = 0.862
res = pd.read_csv(TAB / "D_full_strategy_comparison.csv")
STRATS = ["truncation", "extraction", "chunking"]
TIERS = ["gpt-4o-mini", "gpt-5.4-mini", "gpt-5.4"]
SCOLOR = {"truncation": "#4C78A8", "extraction": "#54A24B", "chunking": "#E45756"}

# =============================================================================
# FIGURE A: strategy AUC (with CIs) + cost — elaborate isn't better, just costly
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: AUC by strategy, grouped by tier, with CI whiskers
ax = axes[0]
x = np.arange(len(TIERS)); w = 0.25
for i, strat in enumerate(STRATS):
    sub = res[res.strategy == strat].set_index("model").reindex(TIERS)
    aucs = sub.ROC_AUC.values
    lo = aucs - sub.ci_low.values; hi = sub.ci_high.values - aucs
    ax.bar(x + (i-1)*w, aucs, w, yerr=[lo, hi], capsize=3,
           label=strat, color=SCOLOR[strat])
ax.axhline(NUMERIC_BASELINE, ls="--", color="#333", lw=1.2)
ax.text(len(TIERS)-0.5, NUMERIC_BASELINE+0.004, "numeric baseline 0.862",
        fontsize=8, ha="right", color="#333", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(TIERS, fontsize=9)
ax.set_ylabel("ROC-AUC (bootstrap 95% CI)"); ax.set_ylim(0.70, 0.98)
ax.set_title("Strategies are statistically indistinguishable\n(CIs overlap within each model)")
ax.legend(fontsize=8.5, loc="lower right", title="strategy")

# Right: cost by strategy (gpt-5.4) — elaborate costs multiples for no gain
ax = axes[1]
s54 = res[res.model == "gpt-5.4"].set_index("strategy").reindex(STRATS)
bars = ax.bar(range(len(STRATS)), s54.cost_usd.values,
              color=[SCOLOR[s] for s in STRATS])
for i, (bar, auc) in enumerate(zip(bars, s54.ROC_AUC.values)):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f"${bar.get_height():.1f}\nAUC {auc:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(STRATS))); ax.set_xticklabels(STRATS, fontsize=9)
ax.set_ylabel("cost (USD, 222 firms, gpt-5.4)")
ax.set_ylim(0, max(s54.cost_usd.values)*1.3)
ax.set_title("...but elaborate strategies cost multiples\n(chunking 4× truncation, same AUC)")

fig.suptitle("Paper D (TDC) — simplicity wins in long-document credit analysis",
             fontsize=12.5)
fig.text(0.01, 0.005, SRC, fontsize=6, color="gray", ha="left")
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.savefig(FIG / "D_simplicity_wins.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  → saved: {rel(FIG / 'D_simplicity_wins.png')}")

# =============================================================================
# FIGURE B: model strength drives performance; best text vs numeric baseline
# =============================================================================
fig, ax = plt.subplots(figsize=(8, 5))
for strat in STRATS:
    sub = res[res.strategy == strat].set_index("model").reindex(TIERS)
    ax.plot(range(len(TIERS)), sub.ROC_AUC.values, "o-", label=strat,
            color=SCOLOR[strat], lw=2, markersize=7)
ax.axhline(NUMERIC_BASELINE, ls="--", color="#333", lw=1.2)
ax.text(0, NUMERIC_BASELINE+0.004, "numeric baseline 0.862",
        fontsize=8.5, color="#333", fontweight="bold")
ax.fill_between([-0.3, len(TIERS)-0.7], 0.862-0.050, 0.862+0.050,
                color="gray", alpha=0.12, label="numeric ±1 SD")
ax.set_xticks(range(len(TIERS))); ax.set_xticklabels(TIERS, fontsize=9)
ax.set_ylabel("ROC-AUC"); ax.set_ylim(0.75, 0.96)
ax.set_xlim(-0.3, len(TIERS)-0.7)
ax.set_title("Model strength — not strategy — drives text-based performance\n"
             "(strongest model beats the numeric baseline)")
ax.legend(fontsize=8.5, loc="lower right")
fig.text(0.01, 0.005, SRC, fontsize=6, color="gray", ha="left")
fig.tight_layout()
fig.savefig(FIG / "D_model_strength.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  → saved: {rel(FIG / 'D_model_strength.png')}")

# =============================================================================
# Summary line for the paper
# =============================================================================
best = res.loc[res.ROC_AUC.idxmax()]
trunc54 = res[(res.strategy=="truncation") & (res.model=="gpt-5.4")].iloc[0]
chunk54 = res[(res.strategy=="chunking") & (res.model=="gpt-5.4")].iloc[0]
print("\n" + "="*66)
print("Paper D (TDC) — summary")
print("="*66)
print(f"  Best: {best.strategy}/{best.model} AUC {best.ROC_AUC:.3f} "
      f"[{best.ci_low:.3f}, {best.ci_high:.3f}]")
print(f"  Truncation vs chunking (gpt-5.4): {trunc54.ROC_AUC:.3f} vs {chunk54.ROC_AUC:.3f} "
      f"— CIs overlap, but chunking costs ${chunk54.cost_usd:.1f} vs ${trunc54.cost_usd:.1f}")
print(f"  Best text ({best.ROC_AUC:.3f}) vs numeric baseline (0.862): "
      f"{'beats' if best.ci_low > 0.862 else 'matches'}")
print(f"  → Simplicity wins: plain truncation + a strong model is the efficient choice.")

  → saved: artifacts\figures\D_simplicity_wins.png
  → saved: artifacts\figures\D_model_strength.png

Paper D (TDC) — summary
  Best: truncation/gpt-5.4 AUC 0.910 [0.870, 0.944]
  Truncation vs chunking (gpt-5.4): 0.910 vs 0.898 — CIs overlap, but chunking costs $17.0 vs $4.2
  Best text (0.910) vs numeric baseline (0.862): beats
  → Simplicity wins: plain truncation + a strong model is the efficient choice.
